<a href="https://www.kaggle.com/code/youssefhazem75/traffic-signs-detection?scriptVersionId=369664809" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Traffic Sign Recognition Using YOLO and Vision Transformers

This project combines a **YOLO** object detection model trained on the **TT100K dataset** for real-time sign detection and spatial localization with a fine-tuned **Vision Transformer (ViT)** for high-precision multi-class sign classification. Designed to assist autonomous vehicles and Advanced Driver Assistance Systems (ADAS), the pipeline ensures rapid and reliable decision-making across complex road scenes. Building on foundational traffic sign benchmark research initiated by Stallkamp et al. (2012), the system addresses key real-world challenges such as detecting small traffic signs, handling class imbalance, and adapting to diverse lighting and environmental conditions.

## Environment Set Up

First let's make sure we have installed and loaded all needed packages.

In [ ]:
!pip install accelerate transformers datasets wandb
!pip install ultralytics 

In [ ]:
import tensorflow as tf
import transformers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import random
import torch
import cv2
from PIL import Image, ImageDraw

# TT100K Dataset Exploration

In [ ]:
import os

DATA_ROOT = "/kaggle/input"

print("Available datasets:")
print(os.listdir(DATA_ROOT))

In [ ]:
DATA_ROOT = "/kaggle/input/datasets"

print("Datasets inside:")
print(os.listdir(DATA_ROOT))

In [ ]:
BRAUNGE_ROOT = "/kaggle/input/datasets/braunge"

print(os.listdir(BRAUNGE_ROOT))

In [ ]:
TT100K_ROOT = "/kaggle/input/datasets/braunge/tt100k"

for root, dirs, files in os.walk(TT100K_ROOT):
    level = root.replace(TT100K_ROOT, "").count(os.sep)
    
    if level <= 3:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")
        
        for file in files[:5]:
            print(f"{indent}    {file}")

## Dataset Configuration

In [ ]:
import yaml

TT100K_ROOT = "/kaggle/input/datasets/braunge/tt100k/mydata"
YAML_PATH = f"{TT100K_ROOT}/YOLOv8_TT100K.yaml"

with open(YAML_PATH, "r") as f:
    config = yaml.safe_load(f)

print("Dataset configuration:\n")

for key, value in config.items():
    print(f"{key}: {value}")

## Dataset Size and Composition

In [ ]:
from pathlib import Path
from collections import Counter

TRAIN_IMG = Path(TT100K_ROOT) / "images/train"
VAL_IMG = Path(TT100K_ROOT) / "images/val"

TRAIN_LABEL = Path(TT100K_ROOT) / "labels/train"
VAL_LABEL = Path(TT100K_ROOT) / "labels/val"

image_extensions = [".jpg", ".jpeg", ".png", ".ppm"]

train_images = [p for p in TRAIN_IMG.iterdir() if p.suffix.lower() in image_extensions]
val_images = [p for p in VAL_IMG.iterdir() if p.suffix.lower() in image_extensions]

train_labels = list(TRAIN_LABEL.glob("*.txt"))
val_labels = list(VAL_LABEL.glob("*.txt"))

print("TT100K Dataset Summary")
print("----------------------")
print(f"Number of classes:     {config['nc']}")
print(f"Training images:       {len(train_images):,}")
print(f"Validation images:     {len(val_images):,}")
print(f"Total images:          {len(train_images) + len(val_images):,}")
print()
print(f"Training label files:  {len(train_labels):,}")
print(f"Validation label files:{len(val_labels):,}")
print(f"Total label files:     {len(train_labels) + len(val_labels):,}")
print()
print("Image formats:", Counter(p.suffix.lower() for p in train_images + val_images))

## Class Distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

class_names = config["names"]
class_counts = Counter()
total_objects = 0

# Read both training and validation annotations
for label_dir in [TRAIN_LABEL, VAL_LABEL]:
    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 5:
                    class_id = int(parts[0])
                    class_counts[class_id] += 1
                    total_objects += 1

print(f"Total annotated traffic signs: {total_objects:,}")
print(f"Number of classes: {len(class_names)}")
print()

print("Objects per class:")
for class_id, name in enumerate(class_names):
    print(f"{class_id:2d} | {name:6s} | {class_counts[class_id]:,}")

# Plot
names = [class_names[i] for i in range(len(class_names))]
counts = [class_counts[i] for i in range(len(class_names))]

plt.figure(figsize=(16, 6))
plt.bar(names, counts)
plt.xlabel("Traffic Sign Class")
plt.ylabel("Number of Annotated Signs")
plt.title("TT100K Class Distribution")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Image Resolution and Format Analysis

In [ ]:
from PIL import Image
from collections import Counter

resolution_counts = Counter()

all_images = train_images + val_images

for img_path in all_images:
    with Image.open(img_path) as img:
        resolution_counts[img.size] += 1

print(f"Total images checked: {len(all_images):,}")
print(f"Number of different resolutions: {len(resolution_counts)}")
print()

print("Resolution distribution:")
for resolution, count in resolution_counts.most_common():
    print(f"{resolution[0]} x {resolution[1]} : {count:,} images")

## Bounding Box Analysis

In [ ]:
import numpy as np

bbox_widths = []
bbox_heights = []
bbox_areas = []

# YOLO format:
# class_id x_center y_center width height
# width and height are normalized between 0 and 1

for label_dir in [TRAIN_LABEL, VAL_LABEL]:
    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) >= 5:
                    w = float(parts[3])
                    h = float(parts[4])

                    # Convert normalized dimensions to pixels
                    width_px = w * 2048
                    height_px = h * 2048

                    bbox_widths.append(width_px)
                    bbox_heights.append(height_px)
                    bbox_areas.append(width_px * height_px)

bbox_widths = np.array(bbox_widths)
bbox_heights = np.array(bbox_heights)
bbox_areas = np.array(bbox_areas)

print(f"Bounding boxes analyzed: {len(bbox_widths):,}")
print()

print("Width (pixels)")
print(f"  Min:    {bbox_widths.min():.1f}")
print(f"  Median: {np.median(bbox_widths):.1f}")
print(f"  Mean:   {bbox_widths.mean():.1f}")
print(f"  Max:    {bbox_widths.max():.1f}")

print("\nHeight (pixels)")
print(f"  Min:    {bbox_heights.min():.1f}")
print(f"  Median: {np.median(bbox_heights):.1f}")
print(f"  Mean:   {bbox_heights.mean():.1f}")
print(f"  Max:    {bbox_heights.max():.1f}")

print("\nBounding-box area")
print(f"  Median: {np.median(bbox_areas):,.1f} pixels²")
print(f"  Mean:   {bbox_areas.mean():,.1f} pixels²")

# Percentage of relatively small signs
small = np.sum((bbox_widths < 32) | (bbox_heights < 32))
print(f"\nBoxes with width OR height < 32 px: {small:,}")
print(f"Percentage: {small / len(bbox_widths) * 100:.2f}%")

## Duplicate Image Analysis

In [ ]:
import hashlib
from collections import defaultdict

hash_to_images = defaultdict(list)

for img_path in all_images:
    with open(img_path, "rb") as f:
        image_hash = hashlib.md5(f.read()).hexdigest()

    hash_to_images[image_hash].append(img_path.name)

duplicate_groups = {
    h: names
    for h, names in hash_to_images.items()
    if len(names) > 1
}

duplicate_image_count = sum(
    len(names) - 1 for names in duplicate_groups.values()
)

print("Duplicate Image Analysis")
print("------------------------")
print(f"Images checked:          {len(all_images):,}")
print(f"Duplicate groups:        {len(duplicate_groups):,}")
print(f"Extra duplicate images: {duplicate_image_count:,}")

if duplicate_groups:
    print("\nExample duplicate groups:")
    for names in list(duplicate_groups.values())[:10]:
        print(names)

## Sample Images and Annotations

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

random.seed(42)

sample_images = random.sample(train_images, 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, img_path in zip(axes.flatten(), sample_images):

    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    label_path = TRAIN_LABEL / f"{img_path.stem}.txt"

    with open(label_path, "r") as f:
        for line in f:
            class_id, xc, yc, w, h = map(float, line.split())

            class_id = int(class_id)

            img_w, img_h = image.size

            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h

            draw.rectangle([x1, y1, x2, y2], width=5)

            draw.text(
                (x1, max(0, y1 - 20)),
                class_names[class_id]
            )

    ax.imshow(image)
    ax.set_title(img_path.name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Annotation Quality Check

In [ ]:
invalid_boxes = []
very_tiny_boxes = []

for label_dir in [TRAIN_LABEL, VAL_LABEL]:
    for label_file in label_dir.glob("*.txt"):
        with open(label_file, "r") as f:
            for line_num, line in enumerate(f, start=1):
                parts = line.strip().split()

                if len(parts) != 5:
                    invalid_boxes.append((label_file.name, line_num, "Wrong format"))
                    continue

                class_id, xc, yc, w, h = map(float, parts)

                # Check class ID
                if not (0 <= int(class_id) < len(class_names)):
                    invalid_boxes.append((label_file.name, line_num, "Invalid class"))

                # Check YOLO normalized coordinates
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    invalid_boxes.append((label_file.name, line_num, "Invalid coordinates"))

                # Extremely tiny: less than 5 px in either dimension
                if w * 2048 < 5 or h * 2048 < 5:
                    very_tiny_boxes.append(
                        (label_file.name, int(class_id), w * 2048, h * 2048)
                    )

print("Annotation Quality Check")
print("------------------------")
print(f"Invalid annotations:       {len(invalid_boxes):,}")
print(f"Extremely tiny boxes:      {len(very_tiny_boxes):,}")

if invalid_boxes:
    print("\nExamples of invalid annotations:")
    for item in invalid_boxes[:10]:
        print(item)

if very_tiny_boxes:
    print("\nExamples of extremely tiny boxes:")
    for item in very_tiny_boxes[:10]:
        print(item)

## Visual Dataset Challenges

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

random.seed(10)
challenge_samples = random.sample(all_images, 12)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, img_path in zip(axes.flatten(), challenge_samples):
    image = Image.open(img_path).convert("RGB")
    ax.imshow(image)
    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.suptitle(
    "TT100K Road Scenes: Lighting, Background and Scene Variation",
    fontsize=16
)
plt.tight_layout()
plt.show()

### Dataset Challenges

The exploration of this TT100K version reveals several important challenges:

- **Class imbalance:** The 50 classes are not equally represented. The most frequent class (`pn`) contains 7,554 instances, while the least frequent (`p19`) contains 535, giving an imbalance ratio of approximately **14.12×**.

- **Small objects:** Traffic signs occupy a very small part of the original 2048×2048 road images. The median bounding box is approximately **36×40 pixels**, and **43.58%** of all annotated signs have a width or height below 32 pixels. This makes small-object detection a major challenge.

- **Background and scene variation:** The dataset contains different road environments including highways, intersections, urban roads, roadside buildings, vegetation, traffic, and different illumination conditions. This increases visual variability around the traffic signs.

- **Occlusion and motion blur:** These may occur in real road scenes, but they are not explicitly annotated by this dataset and were not quantitatively measured in this exploration.

- **Domain shift:** TT100K contains road scenes collected in China. A model intended for deployment in another country, such as Egypt, may encounter different sign designs, road layouts, language, infrastructure, weather, and driving environments. Performance on TT100K therefore may not directly represent performance under the final deployment conditions.

- **Annotation quality and label noise:** All **74,008** inspected annotations followed valid YOLO formatting and contained valid normalized coordinates. Only one extremely small bounding box was found (approximately 2×7 pixels). However, this structural check cannot determine whether every class label is semantically correct, so possible label noise cannot be completely ruled out.

- **Duplicate images:** No exact duplicate images were found among the **24,221** images using file-hash comparison. This check detects exact duplicates but does not rule out visually similar or near-duplicate frames.

In [ ]:
counts = [class_counts[i] for i in range(len(class_names))]

max_class = max(range(len(class_names)), key=lambda i: class_counts[i])
min_class = min(range(len(class_names)), key=lambda i: class_counts[i])

print("Class Imbalance Summary")
print("-----------------------")
print(f"Most frequent class:  {class_names[max_class]} "
      f"({class_counts[max_class]:,} instances)")

print(f"Least frequent class: {class_names[min_class]} "
      f"({class_counts[min_class]:,} instances)")

print(f"Imbalance ratio:      "
      f"{class_counts[max_class] / class_counts[min_class]:.2f}x")

print(f"Mean instances/class: {sum(counts) / len(counts):,.1f}")
print(f"Median instances/class: {np.median(counts):,.1f}")

### YOLO Annotation Format

Each traffic sign is represented by one row in its corresponding label file:

`class_id  x_center  y_center  width  height`

The bounding-box coordinates are normalized relative to the image width and height.

In [ ]:
# Show a real annotation file from TT100K

example_label = train_labels[0]

print("Label file:", example_label.name)
print("\nRaw annotation contents:\n")

with open(example_label, "r") as f:
    lines = f.readlines()

for line in lines:
    class_id, xc, yc, w, h = line.strip().split()

    print("Raw:", line.strip())
    print(f"  Class ID:   {class_id}")
    print(f"  Class name: {class_names[int(class_id)]}")
    print(f"  x_center:   {xc}")
    print(f"  y_center:   {yc}")
    print(f"  width:      {w}")
    print(f"  height:     {h}")
    print()

### Sample from Each Class

The following grid displays one annotated traffic-sign instance from each of the 50 classes. 
Each sample is cropped directly from its ground-truth bounding box.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def crop_yolo_box(image, xc, yc, w, h):
    """Crop a YOLO-format bounding box from a PIL image."""
    
    img_w, img_h = image.size

    x1 = max(0, int((xc - w / 2) * img_w))
    y1 = max(0, int((yc - h / 2) * img_h))
    x2 = min(img_w, int((xc + w / 2) * img_w))
    y2 = min(img_h, int((yc + h / 2) * img_h))

    return image.crop((x1, y1, x2, y2))


# Find one example for each class
class_samples = {}

for label_file in train_labels:
    if len(class_samples) == len(class_names):
        break

    img_path = TRAIN_IMG / f"{label_file.stem}.jpg"

    if not img_path.exists():
        continue

    with open(label_file, "r") as f:
        for line in f:
            class_id, xc, yc, w, h = map(float, line.split())
            class_id = int(class_id)

            if class_id not in class_samples:
                class_samples[class_id] = (
                    img_path,
                    xc, yc, w, h
                )


# Display one crop per class
fig, axes = plt.subplots(5, 10, figsize=(20, 10))

for class_id, ax in enumerate(axes.flatten()):
    if class_id in class_samples:
        img_path, xc, yc, w, h = class_samples[class_id]

        image = Image.open(img_path).convert("RGB")
        crop = crop_yolo_box(image, xc, yc, w, h)

        ax.imshow(crop)
        ax.set_title(
            f"{class_id}: {class_names[class_id]}",
            fontsize=9
        )

    ax.axis("off")

plt.suptitle("One Ground-Truth Sample from Each TT100K Class", fontsize=16)
plt.tight_layout()
plt.show()

print(f"Classes represented: {len(class_samples)}/{len(class_names)}")

## Set Up the yolo Model
the yolo model detects the position of the sign

In [ ]:
import yaml

# TT100K_ROOT (/kaggle/input/...) is read-only. Ultralytics needs `train`/`val`
# to point at real image folders, so we write a fresh YAML into /kaggle/working
# that points at the input folder instead of the broken paths in the original YAML.
YOLO_YAML = "/kaggle/working/tt100k_yolo.yaml"

yolo_cfg = {
    "path": TT100K_ROOT,      # /kaggle/input/datasets/braunge/tt100k/mydata
    "train": "images/train",
    "val": "images/val",
    "nc": len(class_names),
    "names": class_names,
}

with open(YOLO_YAML, "w") as f:
    yaml.safe_dump(yolo_cfg, f, sort_keys=False)

print(open(YOLO_YAML).read())

### Full Training Run (background-safe)

Once the smoke test above finishes cleanly, run this. It trains for up to 300 epochs but
stops early (`patience=30`) once validation mAP stops improving for 30 straight epochs — so
it runs until the model is actually done improving, not until a clock runs out.

**To run this unattended:** click **Save Version → Save & Run All (Commit)** in Kaggle. This
runs the whole notebook on Kaggle's own servers, independent of your browser tab — you can
close the tab or shut your laptop and it keeps going. Check back later on the notebook's
**Logs**/**Output** tabs. Two real caps still apply regardless of "Save & Run All": Kaggle's
per-session GPU runtime limit (check **Settings → Session options**, typically 9-12h), and
your weekly GPU-hour quota.

Because a commit run can still get cut off by that session limit, the cell below pushes
`best.pt`/`last.pt` to Hugging Face every 5 epochs (needs the `HF_TOKEN` secret — see the
markdown cell further down for setup). If a run gets interrupted, download the latest
checkpoint from your HF repo and resume with:
`YOLO("<downloaded>/last.pt").train(resume=True)`.

**One-time Hugging Face setup** (do this before running the training cell below):
1. On huggingface.co, go to Settings → Access Tokens and create a token with **write** access.
2. In this Kaggle notebook, open **Add-ons → Secrets**, add a secret named `HF_TOKEN` with that token, and attach it to the notebook. Internet must be on.
3. Change `HF_REPO` in the next cell to your own username/repo.

In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
from ultralytics import YOLO

login(UserSecretsClient().get_secret("HF_TOKEN"))

HF_REPO = "youssefhazem/yolov8s-tt100k"     # <-- change this

api = HfApi()
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)  # flip private=False to share

def push_checkpoint(trainer):
    # Uploads every 5 epochs so progress survives a Kaggle session being cut off.
    # Never let an upload hiccup kill a multi-hour training run.
    if (trainer.epoch + 1) % 5 != 0:
        return
    try:
        for fname in ("best.pt", "last.pt"):
            p = trainer.save_dir / "weights" / fname
            if p.exists():
                api.upload_file(path_or_fileobj=str(p), path_in_repo=fname, repo_id=HF_REPO,
                                commit_message=f"epoch {trainer.epoch + 1}")
        print(f"[epoch {trainer.epoch + 1}] pushed checkpoint to {HF_REPO}")
    except Exception as e:
        print("HF upload failed (training continues):", e)

yolo_model = YOLO("yolov8s.pt")   # pretrained on COCO; fine-tuned from scratch on TT100K below
yolo_model.add_callback("on_model_save", push_checkpoint)

yolo_model.train(
    data=YOLO_YAML,
    imgsz=1280,           # TT100K signs are small (median ~36x40 px on a 2048 px image);
                          # imgsz=640 would shrink them to ~11 px and hurt detection
    epochs=300,           # high ceiling - patience below will stop it well before this
    patience=30,          # stop if val mAP hasn't improved in 30 epochs
    batch=16,
    device=0,
    workers=4,
    fliplr=0.0,           # direction-specific signs (i2, i4, i5) must not be mirrored
    save_period=5,
    project="/kaggle/working/runs",
    name="tt100k_yolo",
)

### Upload Final Artifacts to Hugging Face

`best.pt`/`last.pt` were already pushed to `HF_REPO` every 5 epochs by the training callback.
This just uploads the remaining run artifacts (metrics, plots, the data config) once training
finishes.

In [ ]:
run_dir = Path("/kaggle/working/runs/tt100k_yolo")
for f in ["results.csv", "args.yaml", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p = run_dir / f
    if p.exists():
        api.upload_file(path_or_fileobj=str(p), path_in_repo=Path(f).name, repo_id=HF_REPO)
        print("uploaded:", f)

api.upload_file(path_or_fileobj=YOLO_YAML, path_in_repo="tt100k.yaml", repo_id=HF_REPO)
print(f"\nDone: https://huggingface.co/{HF_REPO}")

### Downloading latest(best) trained model

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))

HF_REPO = "youssefhazem/yolov8s-tt100k"

downloaded = hf_hub_download(repo_id=HF_REPO, filename="best.pt")

dest = Path("/kaggle/working/runs/tt100k_yolo/weights/best.pt")
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(downloaded, dest)

print("best.pt ready at:", dest)

### Evaluating the model on the Full Validation Set

In [ ]:
import numpy as np

box = metrics.box

# F1 = harmonic mean of precision and recall, per class
f1 = 2 * box.p * box.r / (box.p + box.r + 1e-16)

print("Overall (mean across all 50 classes):")
print(f"  Precision : {box.mp:.4f}")
print(f"  Recall    : {box.mr:.4f}")
print(f"  F1        : {f1.mean():.4f}")
print(f"  mAP50     : {box.map50:.4f}")
print(f"  mAP50-95  : {box.map:.4f}")
print()

print("Per-class Precision / Recall / F1 (worst F1 first - these need attention):")
rows = sorted(zip(box.ap_class_index, box.p, box.r, f1, box.ap50), key=lambda x: x[3])
for class_idx, p, r, f, ap50 in rows:
    print(f"  {class_names[class_idx]:6s}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  AP50={ap50:.3f}")

In [ ]:
best_yolo = YOLO("/kaggle/working/runs/tt100k_yolo/weights/best.pt")

metrics = best_yolo.val(data=YOLO_YAML, imgsz=1280, split="val")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print()
print("Per-class AP50 (lowest first - these are the classes to watch):")
per_class = sorted(zip(metrics.box.ap_class_index, metrics.box.ap50), key=lambda x: x[1])
for class_idx, ap in per_class:
    print(f"  {class_names[class_idx]:6s} AP50={ap:.3f}")

### Test on a Few Sample Images

Visual sanity check: run the trained model on some real validation images and look at the boxes.

In [ ]:
import random
import matplotlib.pyplot as plt

random.seed(7)
test_samples = random.sample(val_images, 6)

results = best_yolo.predict([str(p) for p in test_samples], imgsz=1280, conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, res in zip(axes.flatten(), results):
    ax.imshow(res.plot()[:, :, ::-1])   # BGR -> RGB
    ax.set_title(Path(res.path).name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/yolo_sample_predictions.png", dpi=150)
plt.show()